In [3]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Load all datasets
print("Loading datasets...")

race_df = pd.read_csv('data/processed/race_results_2025-6_clean.csv')
reddit_df = pd.read_csv('data/raw/reddit_posts.csv')
youtube_df = pd.read_csv('data/processed/youtube_engagement_2025-6.csv')
news_df = pd.read_csv('data/processed/news_mentions.csv')

print(f"Race Results: {len(race_df)} rows")
print(f"Reddit Mentions: {len(reddit_df)} rows")
print(f"YouTube Engagement: {len(youtube_df)} rows")
print(f"News Mentions: {len(news_df)} rows")

Loading datasets...
Race Results: 2013 rows
Reddit Mentions: 77 rows
YouTube Engagement: 5 rows
News Mentions: 5 rows


In [4]:
def audit_dataset(df, name):
    """
    Print comprehensive audit of dataset structure.
    """
    print(f"\n{'='*50}")
    print(f"DATASET: {name}")
    print(f"{'='*50}")
    print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
    print(f"\nColumn Names and Types:")
    print("-" * 40)
    for col in df.columns:
        dtype = df[col].dtype
        sample = df[col].dropna().iloc[0] if not df[col].dropna().empty else "N/A"
        print(f"  {col:25} | {str(dtype):10} | Sample: {str(sample)[:30]}")
    print(f"\nMissing Values:")
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(missing[missing > 0])
    else:
        print("  None")
    return None

# Audit each dataset
audit_dataset(race_df, "Race Results")
audit_dataset(reddit_df, "Reddit Mentions")
audit_dataset(youtube_df, "YouTube Engagement")
audit_dataset(news_df, "News Mentions")


DATASET: Race Results
Shape: 2013 rows x 33 columns

Column Names and Types:
----------------------------------------
  year                      | int64      | Sample: 2025
  race_number               | int64      | Sample: 1
  race_name                 | str        | Sample: 2025 Daytona 500
  Race_Date                 | str        | Sample: 2025-02-16
  track                     | str        | Sample: Daytona International Speedway
  track_type                | str        | Sample: road course
  track_miles               | float64    | Sample: 2.4
  total_laps                | float64    | Sample: 95.0
  caution_flags             | int64      | Sample: 8
  caution_laps              | int64      | Sample: 47
  lead_changes              | int64      | Sample: 56
  avg_speed_mph             | float64    | Sample: 129.159
  pole_speed_mph            | float64    | Sample: 182.745
  margin_of_victory         | str        | Sample: .113 sec
  attendance                | float64    | Samp

In [5]:
# Define standard date format
DATE_FORMAT = '%Y-%m-%d'  # ISO 8601: 2024-02-18

def standardize_dates(df, date_columns):
    """
    Convert all date columns to standard format.

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame to standardize
    date_columns : list
        List of column names containing dates

    Returns:
    --------
    pd.DataFrame : DataFrame with standardized dates
    """
    df = df.copy()

    for col in date_columns:
        if col not in df.columns:
            print(f"  Warning: Column '{col}' not found")
            continue

        # Convert to datetime
        df[col] = pd.to_datetime(df[col], errors='coerce')

        # Check for conversion failures
        failed = df[col].isna().sum()
        if failed > 0:
            print(f"  Warning: {failed} dates failed to convert in '{col}'")

        # Convert to standard string format for CSV compatibility
        df[f'{col}_str'] = df[col].dt.strftime(DATE_FORMAT)

    return df

# Apply to each dataset
print("Standardizing dates...")

race_df = standardize_dates(race_df, ['Race_Date'])
reddit_df = standardize_dates(reddit_df, ['race_date'] if 'race_date' in reddit_df.columns else [])
youtube_df = standardize_dates(youtube_df, ['race_date'])
news_df = standardize_dates(news_df, ['race_date'] if 'race_date' in news_df.columns else [])

print("Date standardization complete.")

Standardizing dates...
Date standardization complete.


In [6]:
def standardize_columns(df, column_mapping):
    """
    Rename columns to standard names and convert to lowercase with underscores.

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame to standardize
    column_mapping : dict
        Dictionary mapping old names to new names

    Returns:
    --------
    pd.DataFrame : DataFrame with standardized column names
    """
    df = df.copy()

    # Apply explicit mappings
    df = df.rename(columns=column_mapping)

    # Convert remaining columns to lowercase with underscores
    df.columns = df.columns.str.lower().str.replace(' ', '_')

    return df

# Define standard column names for each dataset
race_columns = {
    'Race_Name': 'race_name',
    'Race_Date': 'race_date',
    'Race_Number': 'race_number',
    'Driver': 'driver',
    'Team': 'team',
    'Sponsor': 'sponsor',
    'Finish_Position': 'finish_position',
    'Laps_Led': 'laps_led'
}

exposure_columns = {
    'race_period': 'race_name',  # If Reddit uses race_period
    'Race_Name': 'race_name',
    'Race_Number': 'race_number',
    'Sponsor': 'sponsor'
}

# Apply standardization
race_df = standardize_columns(race_df, race_columns)
reddit_df = standardize_columns(reddit_df, exposure_columns)
youtube_df = standardize_columns(youtube_df, exposure_columns)
news_df = standardize_columns(news_df, exposure_columns)

print("Column name standardization complete.")

Column name standardization complete.


In [11]:
# add race numbers based on date:
date_to_number = {pd.Timestamp('2025-02-1'): 1,
                  pd.Timestamp('2025-2-12'): 2,
                  pd.Timestamp('2025-2-22'): 3,
                  pd.Timestamp('2025-2-28'): 4,
                  pd.Timestamp('2025-3-8'): 5,
                  pd.Timestamp('2025-3-15'): 6,
                  pd.Timestamp('2025-3-22'): 7,
                  pd.Timestamp('2025-3-30'): 8,
                  pd.Timestamp('2025-4-5'): 9,
                  pd.Timestamp('2025-4-12'): 10,
                  pd.Timestamp('2025-4-26'): 11,
                  pd.Timestamp('2025-5-3'): 12,
                  pd.Timestamp('2025-5-10'): 13,
                  pd.Timestamp('2025-5-17'): 14,
                  pd.Timestamp('2025-5-24'): 15,
                  pd.Timestamp('2025-5-31'): 16,
                  pd.Timestamp('2025-6-7'): 17,
                  pd.Timestamp('2025-6-14'): 18,
                  pd.Timestamp('2025-6-21'): 19,
                  pd.Timestamp('2025-6-27'): 20,
                  pd.Timestamp('2025-7-5'): 21,
                  pd.Timestamp('2025-7-12'): 22,
                  pd.Timestamp('2025-7-19'): 23,
                  pd.Timestamp('2025-7-26'): 24,
                  pd.Timestamp('2025-8-2'): 25,
                  pd.Timestamp('2025-8-9'): 26,
                  pd.Timestamp('2025-8-15'): 27,
                  pd.Timestamp('2025-8-22'): 28,
                  pd.Timestamp('2025-8-30'): 29,
                  pd.Timestamp('2025-9-6'): 30,
                  pd.Timestamp('2025-9-12'): 31,
                  pd.Timestamp('2025-9-20'): 32,
                  pd.Timestamp('2025-9-27'): 33,
                  pd.Timestamp('2025-10-4'): 34,
                  pd.Timestamp('2025-10-12'): 35,
                  pd.Timestamp('2025-10-18'): 36,
                  pd.Timestamp('2025-10-27'): 37,
                  pd.Timestamp('2025-11-1'): 38,
                  pd.Timestamp('2026-2-21'): 39,
                  pd.Timestamp('2026-2-28'): 40,
                  pd.Timestamp('2026-3-7'): 41,
                  pd.Timestamp('2026-3-14'): 42,
                  pd.Timestamp('2026-3-21'): 43,
                  pd.Timestamp('2026-3-28'): 44,
                  pd.Timestamp('2026-4-11'): 45,
                  pd.Timestamp('2026-4-18'): 46,
                  pd.Timestamp('2026-4-25'): 47,
                  pd.Timestamp('2026-5-2'): 48,
                  pd.Timestamp('2026-5-9'): 49,
                  pd.Timestamp('2026-5-16'): 50,
                  pd.Timestamp('2026-5-23'): 51,
                  pd.Timestamp('2026-5-30'): 52,
                  pd.Timestamp('2026-6-6'): 53,
}
race_series = pd.Series(date_to_number).sort_index()

def assign_race_number(df, date_col='published')-> pd.DataFrame:
    """Assign race numbers to a DataFrame based on a date column."""
    df = df.copy()
    df['race_number'] = df[date_col].map(race_series)
    return df
assign_race_number(reddit_df, 'published')

,id,title,link,author,published,content,date,sponsor,race_number
0,t3_1tzvru0,Ross Chastain currently only has TWO top 10 fi...,https://www.reddit.com/r/NASCAR/comments/1tzvr...,/u/Gragson18GOAT,2026-06-08 02:53:08+00:00,"<table> <tr><td> <a href=""https://www.reddit.c...",2026-06-08,Busch Light,NaN
1,t3_1u0q5dj,2026 LASTCAR Cup & Truck Chase standings (Afte...,https://www.reddit.com/r/NASCAR/comments/1u0q5...,/u/TIFUthebestSubreddit,2026-06-09 00:45:36+00:00,"<!-- SC_OFF --><div class=""md""><p>Cup Series C...",2026-06-09,Busch Light,NaN
2,t3_1u1e1lk,Ross Chastain’s Busch Light Lime scheme for Po...,https://www.reddit.com/r/NASCAR/comments/1u1e1...,/u/BuschWhackerReviews,2026-06-09 18:48:48+00:00,"&#32; submitted by &#32; <a href=""https://www....",2026-06-09,Busch Light,NaN
3,t3_1u6wjun,2026 LASTCAR Cup & Xfinity Chase standings (Af...,https://www.reddit.com/r/NASCAR/comments/1u6wj...,/u/TIFUthebestSubreddit,2026-06-15 23:10:14+00:00,"<!-- SC_OFF --><div class=""md""><p>Cup Series C...",2026-06-15,Busch Light,NaN
4,t3_1u6yy3f,Better Look At Ross Chastain’s Kubota “Veteran...,https://www.reddit.com/r/NASCAR/comments/1u6yy...,/u/InsideGuard2106,2026-06-16 00:54:54+00:00,"<table> <tr><td> <a href=""https://www.reddit.c...",2026-06-16,Busch Light,NaN
...,...,...,...,...,...,...,...,...,...
72,t3_1u6dbaf,"The Day After the Races - June 15, 2026",https://www.reddit.com/r/NASCAR/comments/1u6db...,/u/NASCARThreadBot,2026-06-15 11:00:06+00:00,"<!-- SC_OFF --><div class=""md""><p>Welcome to t...",2026-06-15,Love's Travel Stops,NaN
73,t3_1u6wjun,2026 LASTCAR Cup & Xfinity Chase standings (Af...,https://www.reddit.com/r/NASCAR/comments/1u6wj...,/u/TIFUthebestSubreddit,2026-06-15 23:10:14+00:00,"<!-- SC_OFF --><div class=""md""><p>Cup Series C...",2026-06-15,Love's Travel Stops,NaN
74,t3_1uc8805,Single Season First-Time Winners Record in 2026?,https://www.reddit.com/r/NASCAR/comments/1uc88...,/u/Fine-Lifeguard-4234,2026-06-22 02:24:53+00:00,"<!-- SC_OFF --><div class=""md""><p>We&#39;re on...",2026-06-22,Love's Travel Stops,NaN
75,t3_1ucx7r0,"2026 LASTCAR Cup, Xfinity, and Truck Chase sta...",https://www.reddit.com/r/NASCAR/comments/1ucx7...,/u/TIFUthebestSubreddit,2026-06-22 21:13:35+00:00,"<!-- SC_OFF --><div class=""md""><p>Cup Series C...",2026-06-22,Love's Travel Stops,NaN


In [14]:
race_num_to_name = (race_df[['race_number', 'race_name']].drop_duplicates().set_index('race_number')['race_name']).to_dict()

def assign_race_name(df, race_num_col='race_number')-> pd.DataFrame:
    """Assign race names based on race numbers."""
    df = df.copy()
    df['race_name'] = df[race_num_col].map(race_num_to_name)
    return df

assign_race_name(reddit_df, 'race_number')

KeyError: 'race_number'

In [12]:
def create_race_name_mapping(df, race_col='race_name'):
    """
    Create a standardized mapping for race names.
    Returns dict mapping original names to standardized names.
    """
    unique_races = df[race_col].unique()
    print(f"Found {len(unique_races)} unique race names:")
    for race in sorted(unique_races):
        print(f"  - {race}")
    return unique_races

# Check race names in each dataset
print("\n=== Race Names in Race Results ===")
race_names_main = create_race_name_mapping(race_df)

print("\n=== Race Names in Reddit Data ===")
race_names_reddit = create_race_name_mapping(reddit_df)

print("\n=== Race Names in YouTube Data ===")
race_names_youtube = create_race_name_mapping(youtube_df)

print("\n=== Race Names in News Data ===")
race_names_news = create_race_name_mapping(news_df)


=== Race Names in Race Results ===
Found 52 unique race names:
  - 2025 AdventHealth 400
  - 2025 Ambetter Health 400
  - 2025 Autotrader EchoPark Automotive 400
  - 2025 Bank of America ROVAL 400
  - 2025 Bass Pro Shops Night Race
  - 2025 Brickyard 400 Presented by PPG
  - 2025 Coca-Cola 600
  - 2025 Coke Zero Sugar 400
  - 2025 Cook Out 400
  - 2025 Cracker Barrel 400
  - 2025 Cup Series Championship
  - 2025 Daytona 500
  - 2025 EchoPark Automotive Grand Prix
  - 2025 Enjoy Illinois 300
  - 2025 Firekeepers Casino 400
  - 2025 Food City 500
  - 2025 Go Bowling at The Glen
  - 2025 Goodyear 400
  - 2025 Grant Park 165
  - 2025 Hollywood Casino 400
  - 2025 Iowa Corn 350
  - 2025 Jack Links 500
  - 2025 Mobil 1 301
  - 2025 Pennzoil 400
  - 2025 Quaker State 400 available at Walmart
  - 2025 Shriners Childrens 500
  - 2025 South Point 400
  - 2025 Southern 500
  - 2025 Straight Talk Wireless 400
  - 2025 The Great American Getaway 400
  - 2025 Toyota / Save Mart 350
  - 2025 Viva Me

KeyError: 'race_name'